# CNN 1D — Clasificación de Etapas de Sueño
**Proyecto 3 — MAIA Grupo 25**

Este notebook entrena la CNN 1D de Omar sobre los datos preparados por Diego.

**Antes de ejecutar:** Activar GPU:
`Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`

## 1. Verificar GPU

In [ ]:
import torch
print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Clonar repositorio y cambiar a rama

In [ ]:
!git clone https://github.com/amquinteroc1/MAIA-Grupo25-Proyecto.git
%cd MAIA-Grupo25-Proyecto
!git checkout feature/omar-cnn1d
!git pull
!ls

## 3. Instalar dependencias

In [ ]:
!pip install mlflow scikit-learn awscli -q

## 4. Configurar credenciales AWS y bajar datos desde S3

Diego sube los datos con:
```bash
aws s3 cp data_processed/ s3://maia-grupo25-sleep-data/data_processed/ --recursive
```
Luego ejecuta esta celda para bajarlos a Colab.

In [ ]:
import os

# ⚠️ REEMPLAZA con las credenciales AWS CLI que te dio Diego
os.environ['AWS_ACCESS_KEY_ID']     = 'AQUI_TU_ACCESS_KEY_ID'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'AQUI_TU_SECRET_ACCESS_KEY'
os.environ['AWS_SESSION_TOKEN']     = 'AQUI_TU_SESSION_TOKEN'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'

# Verificar acceso al bucket
!aws s3 ls s3://maia-grupo25-sleep-data/data_processed/

In [ ]:
import os

LOCAL_DATA = '/content/MAIA-Grupo25-Proyecto/data_processed'
os.makedirs(LOCAL_DATA, exist_ok=True)

# Bajar solo los archivos necesarios (evitar X_seq que son mas pesados)
archivos = [
    'X_train.npy',
    'X_test.npy',
    'y_train.npy',
    'y_test.npy',
    'groups_train.npy',
    'groups_test.npy',
]

BUCKET = 's3://maia-grupo25-sleep-data/data_processed'

for archivo in archivos:
    dst = f'{LOCAL_DATA}/{archivo}'
    if not os.path.exists(dst):
        print(f'Descargando {archivo}...')
        !aws s3 cp {BUCKET}/{archivo} {dst}
    else:
        size_gb = os.path.getsize(dst) / 1e9
        print(f'Ya existe: {archivo} ({size_gb:.2f} GB)')

print('\n✓ Datos disponibles en Colab')
!ls -lh {LOCAL_DATA}

## 5. Verificar datos (test de Diego)

In [ ]:
%cd /content/MAIA-Grupo25-Proyecto
!python src/data/test_data_cnn.py

## 6. Verificar arquitectura CNN 1D

In [ ]:
%cd /content/MAIA-Grupo25-Proyecto
!python src/models/cnn1d.py

## 7. Entrenamiento de prueba (5 épocas para validar)

In [ ]:
%cd /content/MAIA-Grupo25-Proyecto/src/models
!python train_cnn.py --epochs 5 --batch_size 128 --lr 1e-3

## 8. Entrenamiento completo (30 épocas)

In [ ]:
%cd /content/MAIA-Grupo25-Proyecto/src/models
!python train_cnn.py \
    --epochs 30 \
    --batch_size 128 \
    --lr 1e-3 \
    --dropout 0.5

## 9. Ver resultados MLflow

In [ ]:
import mlflow

mlflow.set_experiment('sleep-stage-cnn1d')
runs = mlflow.search_runs()
cols = [
    'run_id',
    'metrics.best_val_f1_macro',
    'metrics.best_val_f1_n1',
    'params.epochs',
    'params.batch_size',
    'params.lr'
]
print(runs[[c for c in cols if c in runs.columns]].to_string())

## 10. Subir modelo entrenado a S3

In [ ]:
import os

MODEL_PATH = '/content/MAIA-Grupo25-Proyecto/experiments/cnn1d/best_cnn1d.pt'
S3_MODEL   = 's3://maia-grupo25-sleep-data/models/best_cnn1d.pt'

if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / 1e6
    print(f'Subiendo modelo ({size_mb:.1f} MB) a S3...')
    !aws s3 cp {MODEL_PATH} {S3_MODEL}
    print(f'✓ Modelo disponible en: {S3_MODEL}')
else:
    print('ERROR: No se encontró el modelo entrenado')

## 11. Descargar modelo a tu máquina local

Desde tu WSL local ejecuta:
```bash
aws s3 cp s3://maia-grupo25-sleep-data/models/best_cnn1d.pt \
    experiments/cnn1d/best_cnn1d.pt

git add experiments/cnn1d/best_cnn1d.pt
git commit -m 'feat: modelo CNN1D entrenado 30 epochs'
git push origin feature/omar-cnn1d
```